# Step 5 — Offline score fusion (Step A fix attempt)

Uses **existing** `B0_scores.csv` + `P3_scores.csv` from notebook 03.
**No** Wave-U-Net / ECAPA re-scoring.

Goal: find a score-level rule with **SASV-EER &lt; B0** under noise (SNR ≤ 10).

| Rule | Definition |
|------|------------|
| `B0` | raw ECAPA score |
| `P3` | always-enhance score |
| `P2_csv` | embedding-gated scores from notebook 03 |
| `max` | `max(s_B0, s_P3)` per trial |
| `blend_w` | `(1-w)*s_B0 + w*s_P3` for fixed `w` |
| `margin_m` | use `s_P3` only if `s_P3 > s_B0 + m`, else `s_B0` |


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "noise_gated_lib.py").exists():
    ROOT = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent / "sasv_la2019"))

from noise_gated_lib import (
    DEFAULT_SNRS_DB,
    RUNS_DIR,
    ensure_dirs,
    ensure_sasv_on_path,
    eers_from_preds,
    save_json,
    snr_tag,
)

ensure_dirs()
ensure_sasv_on_path()

SPLIT = "dev"
BLEND_WS = (0.1, 0.3, 0.5, 0.7)
MARGINS = (0.0, 0.02, 0.05, 0.10)
NOISE_SNRS = ("snr10db", "snr5db", "snr0db")  # pass bar: beat B0 here

print("RUNS_DIR", RUNS_DIR)
print("SNR folders", [snr_tag(s) for s in DEFAULT_SNRS_DB])


RUNS_DIR D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_noise_gated\runs
SNR folders ['clean', 'snr15db', 'snr10db', 'snr5db', 'snr0db']


### Load paired B0 / P3 (/ P2) scores


In [2]:
JOIN_KEYS = ["speaker_id", "test_utt", "key"]


def load_pair(tag: str, split: str = SPLIT) -> pd.DataFrame:
    folder = RUNS_DIR / f"step3_{split}_{tag}"
    b0_path = folder / "B0_scores.csv"
    p3_path = folder / "P3_scores.csv"
    if not b0_path.exists() or not p3_path.exists():
        raise FileNotFoundError(f"Missing scores under {folder}")
    b0 = pd.read_csv(b0_path)
    p3 = pd.read_csv(p3_path)
    merged = b0.merge(
        p3[JOIN_KEYS + ["score"]].rename(columns={"score": "s_p3"}),
        on=JOIN_KEYS,
        how="inner",
        validate="one_to_one",
    ).rename(columns={"score": "s_b0"})
    p2_path = folder / "P2_scores.csv"
    if p2_path.exists():
        p2 = pd.read_csv(p2_path)
        merged = merged.merge(
            p2[JOIN_KEYS + ["score"]].rename(columns={"score": "s_p2"}),
            on=JOIN_KEYS,
            how="left",
            validate="one_to_one",
        )
    else:
        merged["s_p2"] = np.nan
    if len(merged) != len(b0):
        raise RuntimeError(f"{tag}: join dropped rows ({len(merged)} vs {len(b0)})")
    return merged


paired = {}
for snr in DEFAULT_SNRS_DB:
    tag = snr_tag(snr)
    try:
        paired[tag] = load_pair(tag)
        print(f"{tag}: {len(paired[tag])} trials")
    except FileNotFoundError as exc:
        print("skip", tag, exc)

assert paired, "No step3 score CSVs found — run notebook 03 first."


clean: 29548 trials
snr15db: 29548 trials
snr10db: 29548 trials
snr5db: 29548 trials
snr0db: 29548 trials


### Fusion rules + EER sweep


In [3]:
def fuse_scores(df: pd.DataFrame, rule: str) -> np.ndarray:
    s0 = df["s_b0"].to_numpy(dtype=np.float64)
    s3 = df["s_p3"].to_numpy(dtype=np.float64)
    if rule == "B0":
        return s0
    if rule == "P3":
        return s3
    if rule == "P2_csv":
        if df["s_p2"].isna().any():
            raise ValueError("P2 scores missing")
        return df["s_p2"].to_numpy(dtype=np.float64)
    if rule == "max":
        return np.maximum(s0, s3)
    if rule.startswith("blend_"):
        w = float(rule.split("_", 1)[1])
        return (1.0 - w) * s0 + w * s3
    if rule.startswith("margin_"):
        m = float(rule.split("_", 1)[1])
        return np.where(s3 > s0 + m, s3, s0)
    raise ValueError(f"unknown rule: {rule}")


RULES = [
    "B0",
    "P3",
    "P2_csv",
    "max",
    *[f"blend_{w}" for w in BLEND_WS],
    *[f"margin_{m}" for m in MARGINS],
]

rows = []
for tag, df in paired.items():
    keys = df["key"].tolist()
    for rule in RULES:
        try:
            preds = fuse_scores(df, rule).tolist()
        except ValueError as exc:
            print(f"skip {tag}/{rule}: {exc}")
            continue
        metrics = eers_from_preds(preds, keys)
        rows.append(
            {
                "snr": tag,
                "rule": rule,
                "sasv_eer_percent": metrics["sasv_eer_percent"],
                "sv_eer_percent": metrics["sv_eer_percent"],
                "spf_eer_percent": metrics["spf_eer_percent"],
                "n_trials": len(preds),
            }
        )

results = pd.DataFrame(rows)
results


,snr,rule,sasv_eer_percent,sv_eer_percent,spf_eer_percent,n_trials
0,clean,B0,15.218786,1.248266,17.895587,29548
1,clean,P3,22.021095,7.075472,25.139038,29548
2,clean,P2_csv,15.240165,1.230929,17.994259,29548
3,clean,max,15.357754,1.280323,17.980804,29548
4,clean,blend_0.1,15.717645,1.347709,18.487621,29548
5,clean,blend_0.3,16.922035,1.768377,19.788303,29548
6,clean,blend_0.5,18.598383,2.635229,21.563342,29548
7,clean,blend_0.7,19.726340,4.177898,23.045822,29548
8,clean,margin_0.0,15.357754,1.280323,17.980804,29548
9,clean,margin_0.02,15.265108,1.280323,17.976319,29548


### Matrix: SASV-EER (%) by rule × SNR


In [4]:
pivot = results.pivot_table(index="rule", columns="snr", values="sasv_eer_percent")
order = [snr_tag(s) for s in DEFAULT_SNRS_DB if snr_tag(s) in pivot.columns]
pivot = pivot.reindex(columns=order)
display(pivot.round(3))

out_dir = RUNS_DIR / "step5_offline_fusion"
out_dir.mkdir(parents=True, exist_ok=True)
pivot_path = out_dir / "matrix_sasv_eer_offline_fusion.csv"
results_path = out_dir / "all_rules_metrics.csv"
pivot.to_csv(pivot_path)
results.to_csv(results_path, index=False)
print("wrote", pivot_path)
print("wrote", results_path)


snr,clean,snr15db,snr10db,snr5db,snr0db
rule,,,,,
B0,15.219,16.914,17.136,18.012,19.548
P2_csv,15.240,16.933,18.055,20.081,22.035
P3,22.021,23.111,22.417,22.035,22.488
blend_0.1,15.718,17.514,17.606,18.148,19.744
blend_0.3,16.922,18.757,18.598,18.733,20.172
blend_0.5,18.598,19.840,19.744,19.302,20.699
blend_0.7,19.726,21.077,20.706,20.428,21.141
margin_0.0,15.358,16.981,17.225,17.992,19.612
margin_0.02,15.265,16.914,17.147,18.059,19.445


wrote D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_noise_gated\runs\step5_offline_fusion\matrix_sasv_eer_offline_fusion.csv
wrote D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_noise_gated\runs\step5_offline_fusion\all_rules_metrics.csv


### Pass / fail vs B0 under noise

Pass if **SASV-EER &lt; B0** on **all** of SNR 10 / 5 / 0 (strict Step A bar).


In [5]:
b0 = results[results.rule == "B0"].set_index("snr")["sasv_eer_percent"]
summary_rows = []

for rule in RULES:
    if rule == "B0":
        continue
    sub = results[results.rule == rule].set_index("snr")["sasv_eer_percent"]
    deltas = {}
    beats = {}
    for snr in NOISE_SNRS:
        if snr not in sub.index or snr not in b0.index:
            continue
        delta = float(sub[snr] - b0[snr])  # negative = better than B0
        deltas[snr] = delta
        beats[snr] = delta < 0
    if not beats:
        continue
    pass_all = all(beats.values())
    summary_rows.append(
        {
            "rule": rule,
            "pass_noise_bar": pass_all,
            **{f"delta_{k}": v for k, v in deltas.items()},
            **{f"beats_{k}": v for k, v in beats.items()},
        }
    )

summary = pd.DataFrame(summary_rows).sort_values(
    by=["pass_noise_bar", "delta_snr0db"],
    ascending=[False, True],
)
display(summary)

winners = summary[summary.pass_noise_bar]["rule"].tolist()
decision = {
    "step": "A_offline_score_fusion",
    "split": SPLIT,
    "noise_snrs": list(NOISE_SNRS),
    "pass_criterion": "SASV-EER < B0 on all of snr10/5/0",
    "winners": winners,
    "any_pass": bool(winners),
    "recommendation": (
        "Adopt winning rule as new P2 (score-level); then full re-eval if desired."
        if winners
        else "No rule beats B0 under noise — stop gate tuning; try Step B (enhancer) or accept negative result."
    ),
}
save_json(out_dir / "step_a_decision.json", decision)
print(json.dumps(decision, indent=2))


,rule,pass_noise_bar,delta_snr10db,delta_snr5db,delta_snr0db,beats_snr10db,beats_snr5db,beats_snr0db
8,margin_0.02,False,0.010690,4.689896e-02,-0.103335,False,False,True
9,margin_0.05,False,0.046323,3.563284e-02,-0.028506,False,False,True
10,margin_0.1,False,0.010690,-9.269030e-12,0.039196,False,True,False
2,max,False,0.089082,-2.048648e-02,0.064139,False,True,False
7,margin_0.0,False,0.089082,-2.048648e-02,0.064139,False,True,False
3,blend_0.1,False,0.470353,1.354048e-01,0.195760,False,False,False
4,blend_0.3,False,1.462550,7.207534e-01,0.623575,False,False,False
5,blend_0.5,False,2.608103,1.289909e+00,1.150941,False,False,False
6,blend_0.7,False,3.570410,2.415906e+00,1.592788,False,False,False
1,P2_csv,False,0.919327,2.068462e+00,2.486865,False,False,False


{
  "step": "A_offline_score_fusion",
  "split": "dev",
  "noise_snrs": [
    "snr10db",
    "snr5db",
    "snr0db"
  ],
  "pass_criterion": "SASV-EER < B0 on all of snr10/5/0",
  "winners": [],
  "any_pass": false,
  "recommendation": "No rule beats B0 under noise \u2014 stop gate tuning; try Step B (enhancer) or accept negative result."
}


### Best single-SNR improvements (informational)


In [6]:
best_per_snr = []
for snr in order:
    sub = results[results.snr == snr].copy()
    b0_eer = float(sub.loc[sub.rule == "B0", "sasv_eer_percent"].iloc[0])
    sub["delta_vs_b0"] = sub["sasv_eer_percent"] - b0_eer
    # exclude B0 itself when ranking challengers
    chall = sub[sub.rule != "B0"].sort_values("sasv_eer_percent")
    top = chall.iloc[0]
    best_per_snr.append(
        {
            "snr": snr,
            "b0_sasv_eer": b0_eer,
            "best_rule": top["rule"],
            "best_sasv_eer": float(top["sasv_eer_percent"]),
            "delta_vs_b0": float(top["delta_vs_b0"]),
            "beats_b0": bool(top["delta_vs_b0"] < 0),
        }
    )

best_df = pd.DataFrame(best_per_snr)
display(best_df)
best_df.to_csv(out_dir / "best_rule_per_snr.csv", index=False)
print("wrote", out_dir / "best_rule_per_snr.csv")


,snr,b0_sasv_eer,best_rule,best_sasv_eer,delta_vs_b0,beats_b0
0,clean,15.218786,margin_0.1,15.229475,0.010690,False
1,snr15db,16.913747,margin_0.1,16.913747,0.000000,False
2,snr10db,17.135832,margin_0.02,17.146522,0.010690,False
3,snr5db,18.012400,max,17.991914,-0.020486,True
4,snr0db,19.548176,margin_0.02,19.444840,-0.103335,True


wrote D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_noise_gated\runs\step5_offline_fusion\best_rule_per_snr.csv


### Done

- Outputs under `runs/step5_offline_fusion/`
- If `any_pass: true` → use that rule going forward
- If `any_pass: false` → do **not** re-run full notebook 03 for gate tweaks; go to Step B or weaken the claim
